# Advanced Deep Learning Models

Exploring and comparing multiple advanced architectures to improve emotion recognition performance.

**Goals:**
- Implement improved CNN architectures
- Compare multiple models (VGG-like, residual blocks, advanced)
- Track performance improvements
- Save and compare results
- Analyze model complexity vs performance

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src import model as model_module, train as train_module, evaluate as eval_module

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## Load Data

In [ ]:
# Load preprocessed data
data_dir = os.path.join('..', 'data', 'preprocessed')

X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

with open(os.path.join(data_dir, 'class_weights.json'), 'r') as f:
    class_weights = json.load(f)

print(f"Data loaded: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

## Train Advanced CNN

In [ ]:
# Build advanced CNN
advanced_cnn = model_module.advanced_cnn(input_shape=(48, 48, 1), num_classes=7)
print("Advanced CNN Architecture:")
advanced_cnn.summary()

In [ ]:
# Train advanced model
print("Training advanced CNN model...")
history_advanced = train_module.train_model(
    advanced_cnn,
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=32,
    class_weights=class_weights,
    model_name='advanced_cnn'
)

## Train VGG-like Model

In [ ]:
# Build VGG-like model
vgg_model = model_module.vgg_like_model(input_shape=(48, 48, 1), num_classes=7)
print("VGG-like Architecture:")
vgg_model.summary()

In [ ]:
# Train VGG model
print("Training VGG-like model...")
history_vgg = train_module.train_model(
    vgg_model,
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=32,
    class_weights=class_weights,
    model_name='vgg_model'
)

## Compare Models

In [ ]:
# Compare training histories
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Advanced accuracy
axes[0, 0].plot(history_advanced.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(history_advanced.history['val_accuracy'], label='Val', linewidth=2)
axes[0, 0].set_title('Advanced CNN - Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Advanced loss
axes[0, 1].plot(history_advanced.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(history_advanced.history['val_loss'], label='Val', linewidth=2)
axes[0, 1].set_title('Advanced CNN - Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# VGG accuracy
axes[1, 0].plot(history_vgg.history['accuracy'], label='Train', linewidth=2)
axes[1, 0].plot(history_vgg.history['val_accuracy'], label='Val', linewidth=2)
axes[1, 0].set_title('VGG-like - Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# VGG loss
axes[1, 1].plot(history_vgg.history['loss'], label='Train', linewidth=2)
axes[1, 1].plot(history_vgg.history['val_loss'], label='Val', linewidth=2)
axes[1, 1].set_title('VGG-like - Loss', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model comparison plot saved")

## Evaluate Best Model

In [ ]:
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_per_class_metrics

# Compare test accuracies
print("=" * 50)
print("FINAL MODEL COMPARISON ON TEST SET")
print("=" * 50)

results_advanced = evaluate_model(advanced_cnn, X_test, y_test, model_name='advanced')
print("\n")
results_vgg = evaluate_model(vgg_model, X_test, y_test, model_name='vgg')

# Plot confusion matrices
plot_confusion_matrix(y_test, results_advanced['y_pred'], model_name='advanced')
plot_confusion_matrix(y_test, results_vgg['y_pred'], model_name='vgg')

# Plot per-class metrics
plot_per_class_metrics(y_test, results_advanced['y_pred'], model_name='advanced')
plot_per_class_metrics(y_test, results_vgg['y_pred'], model_name='vgg')